# NYC Taxi Fare Prediction - Complete ML Pipeline

**End-to-End Big Data Machine Learning Project**

This notebook contains the complete pipeline combining all 7 notebooks:
1. Data Ingestion (Bronze Layer)
2. EDA & Data Cleaning (Silver Layer)
3. Feature Engineering (Gold Layer)
4. Model Training (5 Algorithms)
5. Hyperparameter Tuning
6. Scalability Analysis
7. Tableau Data Export

**Runtime:** ~45-60 minutes  
**Requirements:** Google Colab (12GB RAM recommended)

---

## SETUP - Run This First

In [ ]:
# Google Colab Environment Setup
import os

# Check if Java is installed (Colab has it by default)
java_check = !which java
if java_check:
    print("✓ Java already installed")
    java_home = !readlink -f $(which java) | sed 's:/bin/java::'
    os.environ["JAVA_HOME"] = java_home[0]
else:
    !apt-get update -qq
    !apt-get install openjdk-11-jdk-headless -qq -y
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
    print("✓ Java installed")

# Install dependencies
!pip install -q pyspark==3.4.0 pyarrow pandas matplotlib seaborn scikit-learn

# Create directory structure
!mkdir -p data/raw data/bronze data/silver data/gold models results

print("✓ Environment ready!")

---
# PART 1: DATA INGESTION (Bronze Layer)
---

In [ ]:
import glob
import requests
from pathlib import Path
from functools import reduce
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    col, lit, year as spark_year, month as spark_month, 
    sum as spark_sum, when, current_timestamp
)

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("NYC_Taxi_ML_Pipeline") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"✓ Spark {spark.version} initialized")

In [ ]:
# Download NYC Taxi Data (3 months: Jan-Mar 2023)
def download_taxi_data(year_val, month_val, output_dir="data/raw"):
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    filename = f"yellow_tripdata_{year_val}-{month_val:02d}.parquet"
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{filename}"
    output_path = os.path.join(output_dir, filename)
    
    if os.path.exists(output_path):
        size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"✓ {filename} exists ({size_mb:.1f} MB)")
        return output_path
    
    print(f"Downloading {filename}...")
    response = requests.get(url, stream=True, timeout=300)
    response.raise_for_status()
    
    with open(output_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"✓ Downloaded ({size_mb:.1f} MB)")
    return output_path

# Download data
data_files = [download_taxi_data(2023, m) for m in [1, 2, 3]]
total_gb = sum(os.path.getsize(f) for f in data_files) / (1024**3)
print(f"\n✓ Total: {total_gb:.2f} GB")

In [ ]:
# Load and combine data
parquet_files = sorted(glob.glob("data/raw/yellow_tripdata_2023-*.parquet"))
dataframes = []

for idx, file_path in enumerate(parquet_files, 1):
    print(f"[{idx}/{len(parquet_files)}] {os.path.basename(file_path)}")
    df = spark.read.parquet(file_path)
    
    # Standardize schema (INT32 -> INT64)
    for col_name in df.columns:
        if str(df.schema[col_name].dataType) == "IntegerType()":
            df = df.withColumn(col_name, col(col_name).cast("long"))
    
    dataframes.append(df)

df_combined = reduce(DataFrame.unionByName, dataframes)
print(f"✓ Combined: {df_combined.count():,} records")

In [ ]:
# Create Bronze Layer with metadata
df_bronze = df_combined \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit("yellow_tripdata_2023")) \
    .withColumn("data_year", spark_year(col("tpep_pickup_datetime"))) \
    .withColumn("data_month", spark_month(col("tpep_pickup_datetime")))

bronze_path = "data/bronze/nyc_taxi_raw"
df_bronze.write.mode("overwrite").partitionBy("data_year", "data_month").parquet(bronze_path)
print(f"✓ Bronze layer saved")

---
# PART 2: EDA & DATA CLEANING (Silver Layer)
---

In [ ]:
from pyspark.sql.functions import unix_timestamp

# Load Bronze layer
df_bronze = spark.read.parquet(bronze_path)
print(f"Bronze: {df_bronze.count():,} records")

# Calculate trip duration
df_clean = df_bronze.withColumn(
    "trip_duration_minutes",
    (unix_timestamp(col("tpep_dropoff_datetime")) - 
     unix_timestamp(col("tpep_pickup_datetime"))) / 60
)

# Apply data quality filters
df_clean = df_clean.filter(
    (col("fare_amount") > 0) & (col("fare_amount") < 500) &
    (col("trip_distance") > 0) & (col("trip_distance") < 100) &
    (col("trip_duration_minutes") > 1) & (col("trip_duration_minutes") < 180) &
    (col("passenger_count").isNotNull()) &
    (col("passenger_count") > 0) & (col("passenger_count") <= 6)
)

print(f"Silver: {df_clean.count():,} records (after cleaning)")

# Save Silver layer
silver_path = "data/silver/nyc_taxi_clean"
df_clean.write.mode("overwrite").partitionBy("data_year", "data_month").parquet(silver_path)
print("✓ Silver layer saved")

---
# PART 3: FEATURE ENGINEERING (Gold Layer)
---

In [ ]:
from pyspark.sql.functions import hour, dayofweek, month as spark_month
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Load Silver layer
df_silver = spark.read.parquet(silver_path)

# Engineer features
df_features = df_silver \
    .withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_day_of_week", dayofweek(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_month", spark_month(col("tpep_pickup_datetime"))) \
    .withColumn("is_rush_hour", when(
        ((col("pickup_hour") >= 7) & (col("pickup_hour") < 10)) |
        ((col("pickup_hour") >= 16) & (col("pickup_hour") < 19)), 1).otherwise(0)) \
    .withColumn("is_weekend", when(
        (col("pickup_day_of_week") == 1) | (col("pickup_day_of_week") == 7), 1).otherwise(0)) \
    .withColumn("is_airport_trip", when(
        col("PULocationID").isin([132, 138, 1]) | 
        col("DOLocationID").isin([132, 138, 1]), 1).otherwise(0)) \
    .withColumn("avg_speed_mph", 
        col("trip_distance") / (col("trip_duration_minutes") / 60)) \
    .withColumn("avg_speed_mph", when(col("avg_speed_mph").isNull(), 0)
        .when(col("avg_speed_mph") > 100, 100).otherwise(col("avg_speed_mph"))) \
    .withColumn("is_credit_card", when(col("payment_type") == 1, 1).otherwise(0))

print("✓ Features engineered")

In [ ]:
# Assemble and scale features
feature_cols = [
    "trip_distance", "passenger_count", "pickup_hour", 
    "pickup_day_of_week", "pickup_month", "is_rush_hour", 
    "is_weekend", "is_airport_trip", "avg_speed_mph", 
    "is_credit_card", "RatecodeID", "PULocationID", "DOLocationID"
]

assembler = VectorAssembler(
    inputCols=feature_cols, 
    outputCol="features_raw", 
    handleInvalid="skip"
)
df_assembled = assembler.transform(df_features)

scaler = StandardScaler(
    inputCol="features_raw", 
    outputCol="features", 
    withMean=True, 
    withStd=True
)
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

df_gold = df_scaled.withColumnRenamed("fare_amount", "label")

# Save Gold layer
gold_columns = [
    "features", "label", "trip_distance", "passenger_count", 
    "pickup_hour", "is_rush_hour", "is_weekend", "is_airport_trip", 
    "VendorID", "data_year", "data_month"
]

df_gold_final = df_gold.select(gold_columns)
gold_path = "data/gold/nyc_taxi_features"
df_gold_final.write.mode("overwrite").partitionBy("data_year", "data_month").parquet(gold_path)
print(f"✓ Gold layer: {df_gold_final.count():,} records, 13 features")

---
# PART 4: MODEL TRAINING (5 Algorithms)
---

In [ ]:
import time
import pandas as pd
from pyspark.ml.regression import (
    LinearRegression, DecisionTreeRegressor, RandomForestRegressor,
    GBTRegressor, GeneralizedLinearRegression
)
from pyspark.ml.evaluation import RegressionEvaluator

# Load Gold layer and split
df_gold = spark.read.parquet(gold_path)
train_data, test_data = df_gold.randomSplit([0.8, 0.2], seed=42)
train_data.cache()
test_data.cache()

print(f"Train: {train_data.count():,} | Test: {test_data.count():,}")

# Create evaluators
evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")
evaluator_mae = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")

In [ ]:
# Train all 5 PySpark models
results = []

# 1. Linear Regression
print("Training Linear Regression...")
lr = LinearRegression(featuresCol="features", labelCol="label", maxIter=10)
lr_model = lr.fit(train_data)
lr_pred = lr_model.transform(test_data)
results.append([
    'Linear Regression',
    evaluator_rmse.evaluate(lr_pred),
    evaluator_r2.evaluate(lr_pred),
    evaluator_mae.evaluate(lr_pred)
])
print(f"  RMSE: ${results[-1][1]:.2f}")

# 2. Decision Tree
print("Training Decision Tree...")
dt = DecisionTreeRegressor(featuresCol="features", labelCol="label", maxDepth=10)
dt_model = dt.fit(train_data)
dt_pred = dt_model.transform(test_data)
results.append([
    'Decision Tree',
    evaluator_rmse.evaluate(dt_pred),
    evaluator_r2.evaluate(dt_pred),
    evaluator_mae.evaluate(dt_pred)
])
print(f"  RMSE: ${results[-1][1]:.2f}")

# 3. Random Forest
print("Training Random Forest...")
rf = RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=20, maxDepth=10)
rf_model = rf.fit(train_data)
rf_pred = rf_model.transform(test_data)
results.append([
    'Random Forest',
    evaluator_rmse.evaluate(rf_pred),
    evaluator_r2.evaluate(rf_pred),
    evaluator_mae.evaluate(rf_pred)
])
print(f"  RMSE: ${results[-1][1]:.2f}")

# 4. Gradient Boosted Trees
print("Training GBT...")
gbt = GBTRegressor(featuresCol="features", labelCol="label", maxIter=20, maxDepth=5)
gbt_model = gbt.fit(train_data)
gbt_pred = gbt_model.transform(test_data)
results.append([
    'GBT',
    evaluator_rmse.evaluate(gbt_pred),
    evaluator_r2.evaluate(gbt_pred),
    evaluator_mae.evaluate(gbt_pred)
])
print(f"  RMSE: ${results[-1][1]:.2f}")

# 5. Generalized Linear Regression
print("Training GLR...")
glr = GeneralizedLinearRegression(featuresCol="features", labelCol="label", family="gaussian")
glr_model = glr.fit(train_data)
glr_pred = glr_model.transform(test_data)
results.append([
    'GLR',
    evaluator_rmse.evaluate(glr_pred),
    evaluator_r2.evaluate(glr_pred),
    evaluator_mae.evaluate(glr_pred)
])
print(f"  RMSE: ${results[-1][1]:.2f}")

# Display comparison
results_df = pd.DataFrame(results, columns=['Model', 'RMSE', 'R²', 'MAE']).sort_values('RMSE')
print("\n" + "="*60)
print("MODEL COMPARISON (PySpark)")
print("="*60)
print(results_df.to_string(index=False))
print("="*60)

In [ ]:
# Scikit-Learn Baseline (5% sample for comparison)
from sklearn.ensemble import RandomForestRegressor as SklearnRF
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

print("Training Scikit-Learn Baseline (5% sample)...")

# Sample and convert to Pandas (exclude timestamp to avoid conversion errors)
pdf_train = train_data.select("features", "label").sample(fraction=0.05, seed=42).toPandas()
pdf_test = test_data.select("features", "label").sample(fraction=0.05, seed=42).toPandas()

# Extract features from Spark vectors
X_train = pdf_train['features'].apply(lambda x: x.toArray()).tolist()
y_train = pdf_train['label'].values
X_test = pdf_test['features'].apply(lambda x: x.toArray()).tolist()
y_test = pdf_test['label'].values

# Train and evaluate
sklearn_rf = SklearnRF(n_estimators=20, max_depth=10, random_state=42, n_jobs=-1)
sklearn_rf.fit(X_train, y_train)
y_pred = sklearn_rf.predict(X_test)

sklearn_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
sklearn_r2 = r2_score(y_test, y_pred)

print(f"\n✓ Scikit-Learn RF (5% data):  RMSE=${sklearn_rmse:.2f}, R²={sklearn_r2:.4f}")
print(f"  PySpark RF (100% data):      RMSE=${results_df[results_df['Model']=='Random Forest']['RMSE'].values[0]:.2f}")
print("\n→ PySpark handles full dataset efficiently!")

---
# PART 5: HYPERPARAMETER TUNING
---

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

# Tune Random Forest
print("Tuning Random Forest...")
rf = RandomForestRegressor(featuresCol="features", labelCol="label", seed=42)

paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 20]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()

tvs = TrainValidationSplit(
    estimator=rf,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_rmse,
    trainRatio=0.8,
    seed=42
)

rf_model_tuned = tvs.fit(train_data)
best_rf = rf_model_tuned.bestModel
rf_predictions = best_rf.transform(test_data)

rf_rmse = evaluator_rmse.evaluate(rf_predictions)
rf_r2 = evaluator_r2.evaluate(rf_predictions)

print(f"✓ Tuned RF - RMSE: ${rf_rmse:.2f}, R²: {rf_r2:.4f}")

In [ ]:
# Stability Analysis (Perturbation Testing)
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.functions import udf

print("Running Stability Analysis...")
stability_sample = test_data.sample(fraction=0.1, seed=42)
orig_preds = best_rf.transform(stability_sample).select("prediction")

# Add 1% noise to features
def perturb_vector(v, epsilon=0.01):
    arr = v.toArray()
    noise = np.random.normal(0, epsilon, len(arr))
    return Vectors.dense(arr + noise)

perturb_udf = udf(lambda x: perturb_vector(x), VectorUDT())
perturbed_df = stability_sample.withColumn("features", perturb_udf(col("features")))
perturbed_preds = best_rf.transform(perturbed_df).select("prediction")

# Calculate prediction differences
rdd_orig = orig_preds.rdd.map(lambda r: r[0])
rdd_pert = perturbed_preds.rdd.map(lambda r: r[0])
diffs = rdd_orig.zip(rdd_pert).map(lambda x: abs(x[0] - x[1])).collect()

avg_diff = np.mean(diffs)
print(f"✓ Avg prediction change with 1% noise: ${avg_diff:.4f}")
print(f"  → Model is {'STABLE' if avg_diff < 0.10 else 'UNSTABLE'} (threshold: $0.10)")

In [ ]:
# Bootstrap Confidence Intervals
print("Calculating Bootstrap 95% CI...")

# Sample residuals for computational efficiency
residuals = rf_predictions.select(
    col("label") - col("prediction")
).rdd.map(lambda r: r[0]).sample(False, 0.1, 42).collect()

# Bootstrap resampling
bootstrap_rmses = []
for _ in range(1000):
    sample_residuals = np.random.choice(residuals, size=len(residuals), replace=True)
    rmse = np.sqrt(np.mean(sample_residuals**2))
    bootstrap_rmses.append(rmse)

ci_lower = np.percentile(bootstrap_rmses, 2.5)
ci_upper = np.percentile(bootstrap_rmses, 97.5)

print(f"✓ Bootstrap RMSE (1000 iterations):")
print(f"  Mean: ${np.mean(bootstrap_rmses):.2f}")
print(f"  95% CI: [${ci_lower:.2f}, ${ci_upper:.2f}]")

---
# PART 6: SCALABILITY ANALYSIS
---

In [ ]:
# Test performance with different data sizes
print("Running Scalability Analysis...")
data_fractions = [0.1, 0.25, 0.5, 0.75, 1.0]
scalability_results = []

for frac in data_fractions:
    print(f"\nTesting {int(frac*100)}% of data...")
    train_sample = train_data.sample(fraction=frac, seed=42)
    
    start_time = time.time()
    rf_temp = RandomForestRegressor(
        featuresCol="features", 
        labelCol="label", 
        numTrees=10, 
        maxDepth=5
    )
    model_temp = rf_temp.fit(train_sample)
    training_time = time.time() - start_time
    
    pred_temp = model_temp.transform(test_data)
    rmse_temp = evaluator_rmse.evaluate(pred_temp)
    
    scalability_results.append([frac, training_time, rmse_temp])
    print(f"  Time: {training_time:.2f}s, RMSE: ${rmse_temp:.2f}")

scale_df = pd.DataFrame(
    scalability_results, 
    columns=['Data Fraction', 'Training Time (s)', 'RMSE']
)
print("\n" + "="*60)
print("SCALABILITY RESULTS")
print("="*60)
print(scale_df.to_string(index=False))
print("="*60)

---
# PART 7: TABLEAU DATA EXPORT
---

In [ ]:
from pyspark.sql.functions import avg, count

# Export prediction sample (1% for Tableau)
tableau_cols = [
    "label", "prediction", "trip_distance", "passenger_count",
    "pickup_hour", "is_rush_hour", "is_weekend", "is_airport_trip"
]
df_export = rf_predictions.select(tableau_cols).sample(fraction=0.01, seed=42).toPandas()
df_export = df_export.rename(columns={"label": "Actual_Fare", "prediction": "Predicted_Fare"})
df_export.to_csv("results/tableau_predictions.csv", index=False)
print(f"✓ Predictions: {len(df_export):,} records")

# Export hourly statistics
hourly_stats = df_gold.groupBy("pickup_hour").agg(
    avg("label").alias("avg_fare"),
    avg("trip_distance").alias("avg_distance"),
    count("*").alias("trip_count")
).orderBy("pickup_hour").toPandas()
hourly_stats.to_csv("results/tableau_hourly_stats.csv", index=False)
print("✓ Hourly stats exported")

# Export rush hour comparison
rush_stats = df_gold.groupBy("is_rush_hour").agg(
    avg("label").alias("avg_fare"),
    count("*").alias("trip_count")
).toPandas()
rush_stats.to_csv("results/tableau_rush_hour_stats.csv", index=False)
print("✓ Rush hour stats exported")

print("\n✓ All Tableau exports complete!")

---
# SUMMARY
---

## Completed Tasks

1. **Data Ingestion** - Downloaded 3 months NYC Taxi data (>1GB)
2.  **Bronze Layer** - Raw data with metadata
3.  **Silver Layer** - Cleaned data (quality filters applied)
4.  **Gold Layer** - 13 engineered features, scaled for ML
5.  **Model Training** - 5 PySpark algorithms (LR, DT, RF, GBT, GLR)
6.  **Scikit-Learn Baseline** - Single-node comparison (5% sample)
7.  **Hyperparameter Tuning** - Optimized Random Forest
8.  **Stability Analysis** - Perturbation testing
9.  **Statistical Significance** - Bootstrap 95% CI
10. **Scalability Analysis** - Performance vs data size
11. **Tableau Exports** - CSV files for visualization

## Key Results

- **Best Model**: Random Forest (Tuned)
- **Dataset**: >1GB, ~8-9M records
- **Features**: 13 engineered features
- **Architecture**: Medallion (Bronze → Silver → Gold)

## Generated Files

- `data/bronze/` - Raw partitioned data
- `data/silver/` - Cleaned data
- `data/gold/` - ML-ready features
- `results/tableau_*.csv` - Tableau exports

---

**Academic Integrity**: All code is original implementation using PySpark MLlib documentation.

**Citation**: NYC Taxi and Limousine Commission (TLC). (2023). Trip Record Data.  
Retrieved from https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page